# Decay files

EvtGen describes which channels a particle decays into, and with which physics model, in decay files. The default table is {py:data}`.DECAY_FILE`. A user decay file overrides its entries, which is the usual way to study a single channel.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

Creating a {py:class}`.Generator` reads the decay and particle tables and initializes Pythia8, which prints a lot of log output. That output is hidden on this page.

In [ ]:
from pyevtgen import DECAY_FILE, Generator, final_state_momenta

generator = Generator(seed=0)

In the default table, a $D^+$ decays into hundreds of channels. One of them is $D^+ \to K^- \pi^+ \pi^+$ with the `D_DALITZ` model, which describes the resonances in this decay:

In [ ]:
lines = DECAY_FILE.read_text().splitlines()
start = lines.index("Decay D+")
end = lines.index("Enddecay", start)
print("\n".join(line for line in lines[start:end] if "D_DALITZ" in line))

## Forcing a channel

A user decay file only needs the entries that it overrides. Here we write one that forces $D^+ \to K^- \pi^+ \pi^+$ with the `D_DALITZ` model, and load it with {py:meth}`.Generator.read_user_decay_file`. The same file can also be passed to the `user_decay_file` argument of {py:class}`.Generator`.

In [ ]:
directory = Path(mkdtemp())
dalitz_file = directory / "dalitz.dec"
dalitz_file.write_text("""
Decay D+
1.0 K- pi+ pi+ D_DALITZ;
Enddecay
End
""")
generator.read_user_decay_file(dalitz_file)

Reading the file prints EvtGen log output, which is hidden here. All $D^+$ decays now have the same final state:

In [ ]:
dalitz_momenta = final_state_momenta(generator.generate("D+", n=50_000))
dalitz_momenta.shape

Loading another user decay file overrides the entry again. This time, the $D^+$ decays uniformly over phase space with the `PHSP` model:

In [ ]:
phsp_file = directory / "phsp.dec"
phsp_file.write_text("""
Decay D+
1.0 K- pi+ pi+ PHSP;
Enddecay
End
""")
generator.read_user_decay_file(phsp_file)

In [ ]:
phsp_momenta = final_state_momenta(generator.generate("D+", n=50_000))

The Dalitz plots show the difference between the two models. With `D_DALITZ`, the $\bar{K}^*(892)^0$ resonance shows up as a band in the $K^-\pi^+$ invariant mass:

In [ ]:
def mass_squared(p: np.ndarray) -> np.ndarray:
    return p[..., 0] ** 2 - np.sum(p[..., 1:] ** 2, axis=-1)


fig, axes = plt.subplots(ncols=2, figsize=(8, 3.5), sharey=True)
for ax, (title, momenta) in zip(
    axes, [("D_DALITZ", dalitz_momenta), ("PHSP", phsp_momenta)], strict=True
):
    s1 = mass_squared(momenta[:, 0] + momenta[:, 1])
    s2 = mass_squared(momenta[:, 0] + momenta[:, 2])
    ax.hist2d(s1, s2, bins=80, cmin=1, rasterized=True)
    ax.set_title(title)
    ax.set_xlabel(r"$m^2(K^-\pi^+_1)$ [GeV$^2$]")
axes[0].set_ylabel(r"$m^2(K^-\pi^+_2)$ [GeV$^2$]")
fig.tight_layout()
plt.show()

The projections onto the invariant masses make the difference more quantitative. Each event enters twice, once for each $K^-\pi^+$ combination, because the two pions are identical:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
for title, momenta in [("D_DALITZ", dalitz_momenta), ("PHSP", phsp_momenta)]:
    m_kpi = np.sqrt(
        np.concatenate([
            mass_squared(momenta[:, 0] + momenta[:, 1]),
            mass_squared(momenta[:, 0] + momenta[:, 2]),
        ])
    )
    ax.hist(m_kpi, bins=100, histtype="step", density=True, label=title)
ax.axvline(0.8955, color="gray", linestyle="dotted", label=r"$\bar{K}^*(892)^0$")
ax.set_xlabel(r"$m(K^-\pi^+)$ [GeV]")
ax.legend()
plt.show()

## Main decay tables

The `decay_file` argument of {py:class}`.Generator` replaces the default table entirely, and accepts several files that are read in order. The `particle_table` argument does the same for the particle properties. Both can only be set when the generator is created.

:::{warning}
EvtGen terminates the process when it encounters a malformed decay file or redefines an existing alias. Python cannot catch this as an exception.
:::